In [1]:
#importing required libraries
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV

In [2]:
# Telling the notebook what the CSV looks like.
TS_COL   = "ts_event"      # timestamp column in the CSV
SYM_COL  = "symbol"        # ticker column
BID_SZ   = "bid_sz_{:02d}" # pattern for bid size columns (00, 01, …)
ASK_SZ   = "ask_sz_{:02d}" # pattern for ask size columns
BID_PX_0 = "bid_px_00"     # best bid price column
ASK_PX_0 = "ask_px_00"     # best ask price column

MAX_LEVELS = 10            # we have 10 depth levels (0–9)
CSV_FILE  = Path("first_25000_rows.csv")  # the data file
BAR_FREQ  = "60S"          # turn events into 60-second bars


In [3]:
# Read the CSV into a pandas table.
raw = pd.read_csv(CSV_FILE)

# Convert the timestamp text into real date-time objects.
raw[TS_COL] = pd.to_datetime(raw[TS_COL], utc=True)

# Sort rows by symbol name and time so earlier events come first.
raw = raw.sort_values([SYM_COL, TS_COL])

# To look at the first 5 rows to make sure it looks right.
raw.head()


,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
0,2024-10-21T11:54:29.221230963Z,2024-10-21 11:54:29.221064336+00:00,10,2,38,C,B,1,233.62,2,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
1,2024-10-21T11:54:29.223936626Z,2024-10-21 11:54:29.223769812+00:00,10,2,38,A,B,0,233.67,2,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
2,2024-10-21T11:54:29.225196809Z,2024-10-21 11:54:29.225030400+00:00,10,2,38,A,B,0,233.67,3,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
3,2024-10-21T11:54:29.712600612Z,2024-10-21 11:54:29.712434212+00:00,10,2,38,A,B,2,233.52,200,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
4,2024-10-21T11:54:29.764839221Z,2024-10-21 11:54:29.764673165+00:00,10,2,38,C,B,2,233.52,200,...,155,1,7,233.25,234.13,55,400,2,1,AAPL


In [4]:
# We need to calculate "how much did the bid size change since the previous event?"
# This loop builds those differences for each depth level.

diff_cols = {}
grouped = raw.groupby(SYM_COL, sort=False)   # handle each stock separately

for k in range(MAX_LEVELS):
    # Change in bid size at this level
    diff_cols[f"d_{BID_SZ.format(k)}"] = grouped[BID_SZ.format(k)].diff().fillna(0)
    # Change in ask size at this level
    diff_cols[f"d_{ASK_SZ.format(k)}"] = grouped[ASK_SZ.format(k)].diff().fillna(0)

# Putting all the new differneces into one DataFrame.
diffs = pd.DataFrame(diff_cols, index=raw.index)
diffs.head()


,d_bid_sz_00,d_ask_sz_00,d_bid_sz_01,d_ask_sz_01,d_bid_sz_02,d_ask_sz_02,d_bid_sz_03,d_ask_sz_03,d_bid_sz_04,d_ask_sz_04,d_bid_sz_05,d_ask_sz_05,d_bid_sz_06,d_ask_sz_06,d_bid_sz_07,d_ask_sz_07,d_bid_sz_08,d_ask_sz_08,d_bid_sz_09,d_ask_sz_09
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,-200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
# Best-level OFI = (change in bid size at level 0) minus (change in ask size at level 0)
ofi_best_event = (diffs["d_bid_sz_00"] - diffs["d_ask_sz_00"]).rename("ofi_best")

# Turn those event-by-event numbers into 60-second sums.
ofi_best_bar = ofi_best_event.set_axis(raw[TS_COL]).resample(BAR_FREQ, label="right").sum()
# First 5 timestamps
ofi_best_bar.head()


/var/folders/cn/7x9ywsx57z53ylw3msdtvjc80000gn/T/ipykernel_94868/533097687.py:5: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ofi_best_bar = ofi_best_event.set_axis(raw[TS_COL]).resample(BAR_FREQ, label="right").sum()


ts_event
2024-10-21 11:55:00+00:00      5.0
2024-10-21 11:56:00+00:00   -317.0
2024-10-21 11:57:00+00:00    199.0
2024-10-21 11:58:00+00:00   -325.0
2024-10-21 11:59:00+00:00    492.0
Freq: 60s, Name: ofi_best, dtype: float64

In [6]:
#Multi-Level OFI
# For every depth level, compute bid-change minus ask-change, same as above.
multi = pd.DataFrame({
    f"ofi_L{k}": diffs[f"d_{BID_SZ.format(k)}"] - diffs[f"d_{ASK_SZ.format(k)}"]
    for k in range(MAX_LEVELS)
})

# Resample so each column shows the total change over every 60-second bar.
multi_bar = multi.set_axis(raw[TS_COL]).resample(BAR_FREQ, label="right").sum()
multi_bar.head()# first 5 timestamps


/var/folders/cn/7x9ywsx57z53ylw3msdtvjc80000gn/T/ipykernel_94868/4201580449.py:9: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  multi_bar = multi.set_axis(raw[TS_COL]).resample(BAR_FREQ, label="right").sum()


,ofi_L0,ofi_L1,ofi_L2,ofi_L3,ofi_L4,ofi_L5,ofi_L6,ofi_L7,ofi_L8,ofi_L9
ts_event,,,,,,,,,,
2024-10-21 11:55:00+00:00,5.0,190.0,-190.0,-30.0,4.0,-175.0,156.0,-111.0,-245.0,399.0
2024-10-21 11:56:00+00:00,-317.0,-197.0,248.0,142.0,94.0,-100.0,100.0,-10.0,-45.0,-54.0
2024-10-21 11:57:00+00:00,199.0,0.0,-200.0,0.0,-4.0,0.0,0.0,1.0,0.0,0.0
2024-10-21 11:58:00+00:00,-325.0,-193.0,142.0,-111.0,-265.0,275.0,-85.0,-36.0,401.0,-99.0
2024-10-21 11:59:00+00:00,492.0,441.0,-191.0,-69.0,173.0,-175.0,-15.0,46.0,-356.0,153.0


In [7]:
#Integrated OFI
# Use PCA to find one smart weighted average of the 10 depth levels.

clean = multi_bar.fillna(0)                 # replace missing values with 0
pca = PCA(n_components=1).fit(clean.values) # fit PCA on the 10 columns
weights = pca.components_[0]                # the 10 weights from PCA
weights = weights / weights.sum()           # scale so all weights add up to 1

# Multiply each level by its weight and add them up.
ofi_integrated = clean.mul(weights, axis=1).sum(1).rename("ofi_integrated")
ofi_integrated.head()# first 5 values


ts_event
2024-10-21 11:55:00+00:00    -214.153105
2024-10-21 11:56:00+00:00    1623.640530
2024-10-21 11:57:00+00:00    -280.179376
2024-10-21 11:58:00+00:00   -2080.854850
2024-10-21 11:59:00+00:00     743.781588
Freq: 60s, Name: ofi_integrated, dtype: float64

In [8]:
# In the dataset you provided (csv file), there were only 5000 rows instead of 25000 and only one ticker (AAPL).
# Thus computing Cross-Asset OFI is not possible
if raw[SYM_COL].nunique() == 1:
    # Only one stock in the CSV — nothing to cross with.
    print("Single-ticker file detected, thus skipping cross-asset OFI")
    ofi_cross = pd.DataFrame(index=ofi_integrated.index)   # empty placeholder
else:
    mid = (raw[BID_PX_0] + raw[ASK_PX_0]) / 2
    mid.index = raw[TS_COL]
    mid_bar = mid.groupby(raw[SYM_COL]).apply(
        lambda s: s.resample(BAR_FREQ, label="right").last()
    ).unstack(level=0)
    rets = mid_bar.pct_change().fillna(0)
    ofi_mat = ofi_integrated.groupby(raw[SYM_COL]).apply(
        lambda s: s.resample(BAR_FREQ, label="right").sum()
    ).unstack(level=0).reindex_like(rets).fillna(0)
    from sklearn.linear_model import LassoCV
    cross_cols = {}
    for sym in rets.columns:
        y = rets[sym].values
        X = ofi_mat.drop(columns=sym).values
        model = LassoCV(cv=5, fit_intercept=True, max_iter=4000).fit(X, y)
        beta = model.coef_
        cross_cols[f"ofi_cross_{sym}"] = (ofi_mat.drop(columns=sym) * beta).sum(1)

    ofi_cross = pd.DataFrame(cross_cols)

ofi_cross.head()# first 5 timestamps


Single-ticker file detected, thus skipping cross-asset OFI


""
ts_event
2024-10-21 11:55:00+00:00
2024-10-21 11:56:00+00:00
2024-10-21 11:57:00+00:00
2024-10-21 11:58:00+00:00
2024-10-21 11:59:00+00:00


In [9]:
#Integrating all the feature columns side-by-side.
final = pd.concat([ofi_best_bar, multi_bar, ofi_integrated, ofi_cross], axis=1)
# Saving to a Parquet file (fast, small, keeps data types).
final.to_parquet("ofi_features.parquet")
final.head()# first 5 timestamps


,ofi_best,ofi_L0,ofi_L1,ofi_L2,ofi_L3,ofi_L4,ofi_L5,ofi_L6,ofi_L7,ofi_L8,ofi_L9,ofi_integrated
ts_event,,,,,,,,,,,,
2024-10-21 11:55:00+00:00,5.0,5.0,190.0,-190.0,-30.0,4.0,-175.0,156.0,-111.0,-245.0,399.0,-214.153105
2024-10-21 11:56:00+00:00,-317.0,-317.0,-197.0,248.0,142.0,94.0,-100.0,100.0,-10.0,-45.0,-54.0,1623.640530
2024-10-21 11:57:00+00:00,199.0,199.0,0.0,-200.0,0.0,-4.0,0.0,0.0,1.0,0.0,0.0,-280.179376
2024-10-21 11:58:00+00:00,-325.0,-325.0,-193.0,142.0,-111.0,-265.0,275.0,-85.0,-36.0,401.0,-99.0,-2080.854850
2024-10-21 11:59:00+00:00,492.0,492.0,441.0,-191.0,-69.0,173.0,-175.0,-15.0,46.0,-356.0,153.0,743.781588
